[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/inference-stacks/inference_quickstart.ipynb)

# Inference with the Self-Improving Models

Call hosted models and your inference stacks: generate responses, stream output, keep conversation state, ground answers in your data, and use tools.

**Docs:** [Quickstart](https://docs.adaptionlabs.ai/inference-stacks/quickstart/) · [Generate a response](https://docs.adaptionlabs.ai/inference-stacks/generate-a-model-response/) · [Key concepts](https://docs.adaptionlabs.ai/inference-stacks/key-concepts/)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

In [ ]:
%pip install -q adaption

## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

> The Responses API requires `adaption >= 0.16.0`. The install cell pulls the latest release.

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

In [ ]:
# A hosted model to call directly.
# See https://docs.adaptionlabs.ai/inference-stacks/models/ for the full list.
MODEL = "qwen3.5-397b"

# Optional: a dataset with a ready retrieval index, to try file_search in Step 5.
FILE_SEARCH_DATASET_ID = None

## Step 1 — List available models

Model availability can vary by API key. `type="hosted"` lists hosted models without inference stacks.

In [ ]:
for model in client.models.list(type="hosted").data:
    print(model.id)

## Step 2 — Generate a response

`output_text` joins the text from every message in the response's `output` array.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Write a one-sentence bedtime story about a unicorn.",
)
print(response.output_text)

Guide the model's behavior with `instructions`:

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="You are a concise technical writer.",
    input="Explain what an API rate limit is.",
)
print(response.output_text)

## Step 3 — Continue a conversation

Each request is independent. To keep context, append the response's `output` to your input list and add the next user turn.

In [ ]:
from typing import Any

history: list[Any] = [
    {"role": "user", "content": "Tell me a joke about databases."},
]
response = client.responses.create(model=MODEL, input=history)
print(response.output_text)

history += response.output
history.append({"role": "user", "content": "Explain why it's funny."})
second = client.responses.create(model=MODEL, input=history)
print(second.output_text)

Or let Adaption store the thread. Set `store=True`, then pass the id as `previous_response_id`. Responses are **not** stored by default and are retained for 30 days.

In [ ]:
first = client.responses.create(
    model=MODEL,
    input="Tell me a joke about databases.",
    store=True,
)
print(first.output_text)

followup = client.responses.create(
    model=MODEL,
    previous_response_id=first.id,
    input="Explain why it's funny.",
    store=True,
)
print(followup.output_text)

## Step 4 — Stream output

Pass `stream=True` to receive server-sent events as the response is generated.

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input="Write a short poem about the ocean.",
    stream=True,
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print()

## Step 5 — Ground answers with file search

The hosted `file_search` tool retrieves relevant rows from your indexed datasets, passes them to the model, and adds citations. Index a dataset's text columns in the app first, then set `FILE_SEARCH_DATASET_ID` above.

In [ ]:
if FILE_SEARCH_DATASET_ID:
    response = client.responses.create(
        model=MODEL,
        input="What return window do we offer on electronics?",
        tools=[{"type": "file_search", "dataset_ids": [FILE_SEARCH_DATASET_ID]}],
    )
    print(response.output_text)
else:
    print(
        "Set FILE_SEARCH_DATASET_ID to a dataset with a ready retrieval index "
        "to try file search."
    )

## Step 6 — Call a function tool

Function tools run in your code. The model emits a `function_call`; you run the function and send the result back as a `function_call_output`, then ask again.

In [ ]:
import json

tools: list[Any] = [
        {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    }
]


def get_weather(city: str) -> str:
    return f"It is 18C and sunny in {city}."


convo: list[Any] = [{"role": "user", "content": "What's the weather in Paris?"}]
response = client.responses.create(model=MODEL, input=convo, tools=tools)

convo += response.output
for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments or "{}")
        convo.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": get_weather(**args),
            }
        )

final = client.responses.create(model=MODEL, input=convo, tools=tools)
print(final.output_text)

## Call one of your stacks

A stack bundles a model with its tools, datasets, and memory. Call it by slug and the request inherits those defaults; request parameters can still override them.

```python
response = client.responses.create(model="stack:your-stack-slug", input="Hello")
print(response.output_text)
```

## Clean up

Stored responses are retained for 30 days. Delete them sooner if you like:

In [ ]:
# client.responses.delete(first.id)
# client.responses.delete(followup.id)